# R.O.A.D. Historical HTR - Colab Training Pipeline

**Model:** Qwen2-VL-7B-Instruct  
**Optimized for:** Google Colab with A100 GPU  
**Setup Time:** 5-10 minutes (dataset download + dependencies)  
**Training Time:** 6-8 hours (A100), 12-15 hours (T4)  

---

**⚠️ IMPORTANT: Set Runtime to GPU (A100 recommended)**

1. Click `Runtime` → `Change runtime type`
2. Set `Hardware accelerator` → `GPU`
3. Set `GPU type` → `A100 40GB` (Colab Pro) or `A100 80GB` (Colab Pro+)
4. Click `Save`

---

**For T4 Users (Free/Pro tier):**  
Training is possible but slower. Uncomment cell 6 to auto-adjust batch size for 16GB VRAM.

**Features:**
- ⚡ Fast setup with pre-built Flash Attention wheels
- 🔄 Git pull support for code updates
- 📊 Automatic validation and visualization
- 💾 Optional Google Drive integration

## 0. Verify GPU & Environment

In [ ]:
# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")
else:
    print("\n❌ No GPU detected! Change runtime type to GPU.")

## 1. (Optional) Mount Google Drive

Mount Drive to save checkpoints persistently. Skip if you'll download them after training.

In [ ]:
# Uncomment to mount Drive
# from google.colab import drive
# drive.mount('/content/drive')

# Set working directory in Drive (optional)
# %cd /content/drive/MyDrive/ROAD

print("✓ Skipping Drive mount (checkpoints will be in /content/)")

## 2. Clone Repository

In [ ]:
import os

# Update with your GitHub repo URL
REPO_URL = "YOUR_GITHUB_REPO_URL"  # e.g., "https://github.com/username/ROAD.git"
REPO_NAME = "ROAD"

if not os.path.exists(REPO_NAME):
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL}
    print(f"✓ Cloned {REPO_NAME}")
else:
    print(f"✓ Repository {REPO_NAME} already exists")

%cd {REPO_NAME}

## 2.1. Update Repository (Optional)

**Run this cell if:**
- You've pushed code updates to GitHub and want to pull them
- Running the notebook again after making changes
- Want to get the latest bug fixes or improvements

**Skip if:**
- First time running (repo was just cloned)

In [ ]:
# Pull latest changes from GitHub
import os

REPO_NAME = "ROAD"

# Ensure we're in the repo directory
if not os.path.exists(".git"):
    if os.path.exists(REPO_NAME):
        print(f"📁 Moving to {REPO_NAME} directory...")
        %cd {REPO_NAME}
    else:
        print("❌ Repository not found. Run cell 2 (Clone Repository) first.")
        raise FileNotFoundError(f"{REPO_NAME} directory not found")

# Check if it's a git repo
if os.path.exists(".git"):
    print("=" * 70)
    print("📥 Pulling Latest Changes")
    print("=" * 70)
    
    # Show current branch
    !echo "\nCurrent branch:"
    !git branch --show-current
    
    # Show status
    !echo "\nStatus before pull:"
    !git status --short
    
    # Stash any local changes (shouldn't be any, but just in case)
    !git stash save "Auto-stash before pull" 2>/dev/null || true
    
    # Pull updates
    !echo "\nPulling from remote..."
    !git pull origin main
    
    # Show recent commits
    print("\n" + "─" * 70)
    print("Recent commits:")
    print("─" * 70)
    !git log --oneline --graph -5
    
    print("\n" + "=" * 70)
    print("✓ Repository updated successfully")
    print("=" * 70)
else:
    print("❌ Not a git repository. Clone first using cell 2.")

## 3. Download Image Dataset

Downloads 5,472 historical document images (~2-3 GB) from Google Cloud Storage.

In [ ]:
## 4. Install Dependencies

Installs Qwen2-VL, PEFT, Flash Attention (optional), and training dependencies.

**Installation time:** 1-3 minutes with pre-built Flash Attention wheels

**Flash Attention is optional:**
- Improves speed (20-30%) and memory efficiency
- Training works without it (uses standard attention)
- Auto-fallback if installation fails or not available

%cd src/qwen2vl

import torch
import sys

# ============================================================================
# INSTALL DEPENDENCIES
# ============================================================================
print("=" * 70)
print("📦 Installing Dependencies")
print("=" * 70)

# Get system info
torch_version = torch.__version__.split("+")[0]
cuda_version = torch.version.cuda.replace(".", "") if torch.version.cuda else ""
python_version = f"{sys.version_info.major}{sys.version_info.minor}"

print(f"\nSystem Configuration:")
print(f"  • PyTorch: {torch_version}")
print(f"  • CUDA: {cuda_version}")
print(f"  • Python: {python_version}")

# ============================================================================
# [0/3] FIX TORCHAO COMPATIBILITY
# ============================================================================
print(f"\n{'─' * 70}")
print("[0/3] Fixing torchao compatibility...")
print(f"{'─' * 70}")

# Upgrade or remove incompatible torchao version
try:
    import torchao
    current_version = torchao.__version__
    print(f"Current torchao version: {current_version}")
    
    # PEFT requires torchao >= 0.16.0
    if tuple(map(int, current_version.split('.')[:2])) < (0, 16):
        print("Upgrading torchao to compatible version...")
        !pip install -q --upgrade torchao
        print("✓ torchao upgraded")
except ImportError:
    print("✓ torchao not installed (optional)")
except Exception as e:
    print(f"⚠️  Could not upgrade torchao: {e}")
    print("  Uninstalling torchao (it's optional)...")
    !pip uninstall -y -q torchao
    print("✓ torchao removed")

# ============================================================================
# [1/3] FLASH ATTENTION (Optional - improves speed/memory)
# ============================================================================
print(f"\n{'─' * 70}")
print("[1/3] Installing Flash Attention (optional)...")
print(f"{'─' * 70}")

# Wheel mappings (PyTorch x CUDA x Python)
wheel_map = {
    # PyTorch 2.4
    ("2.4", "124", "310"): ("flash_attn-2.6.3+cu124torch2.4cxx11abiFALSE-cp310-cp310-linux_x86_64.whl", "v2.6.3"),
    ("2.4", "123", "310"): ("flash_attn-2.6.3+cu123torch2.4cxx11abiFALSE-cp310-cp310-linux_x86_64.whl", "v2.6.3"),
    ("2.4", "121", "310"): ("flash_attn-2.6.3+cu121torch2.4cxx11abiFALSE-cp310-cp310-linux_x86_64.whl", "v2.6.3"),
    
    # PyTorch 2.5
    ("2.5", "124", "310"): ("flash_attn-2.7.0.post2+cu124torch2.5cxx11abiFALSE-cp310-cp310-linux_x86_64.whl", "v2.7.0.post2"),
    ("2.5", "121", "310"): ("flash_attn-2.7.0.post2+cu121torch2.5cxx11abiFALSE-cp310-cp310-linux_x86_64.whl", "v2.7.0.post2"),
    
    # PyTorch 2.6+
    ("2.6", "124", "310"): ("flash_attn-2.7.0.post2+cu124torch2.5cxx11abiFALSE-cp310-cp310-linux_x86_64.whl", "v2.7.0.post2"),
}

# Match wheel
key = (torch_version[:3], cuda_version[:3], python_version)
wheel_info = wheel_map.get(key)

# Fallback for newer CUDA versions
if not wheel_info and len(cuda_version) >= 2:
    for fallback_cuda in ["124", "123", "121"]:
        key_fallback = (torch_version[:3], fallback_cuda, python_version)
        wheel_info = wheel_map.get(key_fallback)
        if wheel_info:
            print(f"⚠️  Using CUDA {fallback_cuda} wheel (closest match)")
            break

flash_installed = False

if wheel_info:
    wheel_name, release_version = wheel_info
    wheel_url = f"https://github.com/Dao-AILab/flash-attention/releases/download/{release_version}/{wheel_name}"
    
    try:
        print(f"✓ Found pre-built wheel")
        print(f"  Downloading from GitHub releases...")
        !wget -q --show-progress {wheel_url}
        
        print(f"  Installing wheel...")
        !pip install -q --no-dependencies {wheel_name}
        !rm -f {wheel_name}
        
        print(f"✓ Flash Attention installed (~30 seconds)")
        flash_installed = True
    except Exception as e:
        print(f"⚠️  Pre-built wheel installation failed: {e}")
        print(f"  Skipping Flash Attention (training will use standard attention)")
else:
    print(f"⚠️  No pre-built wheel for this configuration")
    print(f"  You can:")
    print(f"    • Continue without Flash Attention (works fine, just slower)")
    print(f"    • Or uncomment below to compile from source (10-15 min)")
    print(f"  ")
    # Uncomment to compile from source:
    # !pip install -q flash-attn --no-build-isolation
    print(f"  Skipping Flash Attention for now...")

if not flash_installed:
    print(f"\n💡 Note: Training will work fine without Flash Attention")
    print(f"   It will use standard attention (20-30% slower, same accuracy)")

# ============================================================================
# [2/3] CORE DEPENDENCIES
# ============================================================================
print(f"\n{'─' * 70}")
print("[2/3] Installing core dependencies...")
print(f"{'─' * 70}")

!pip install -q transformers accelerate peft datasets pandas numpy pillow scikit-learn pyyaml tqdm jiwer evaluate

print(f"✓ Core dependencies installed")

# ============================================================================
print(f"\n{'=' * 70}")
if flash_installed:
    print("✓ All dependencies installed (with Flash Attention)")
else:
    print("✓ All dependencies installed (without Flash Attention)")
print(f"{'=' * 70}\n")

In [ ]:
%cd src/qwen2vl

print("📦 Installing dependencies (3-5 minutes)...")
!pip install -q -r requirements.txt

print("\n✓ Dependencies installed")

## 5. Verify Setup

In [ ]:
import torch
import pandas as pd
from pathlib import Path

print("=" * 70)
print("🔍 SETUP VERIFICATION")
print("=" * 70)

# GPU
print("\n[GPU]")
print(f"  Device: {torch.cuda.get_device_name(0)}")
print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"  BF16: {'✓' if torch.cuda.is_bf16_supported() else '✗'}")

# Dataset
print("\n[Dataset]")
train_csv = Path("../../dataset/Train.csv")
test_csv = Path("../../dataset/Test.csv")
image_dir = Path("../../dataset/images")

train_df = pd.read_csv(train_csv)
test_df = pd.read_csv(test_csv)
num_images = len(list(image_dir.glob("*.jpg")))

print(f"  Train samples: {len(train_df)}")
print(f"  Test samples: {len(test_df)}")
print(f"  Images: {num_images}")

# Config
import yaml
with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

print("\n[Config]")
print(f"  Model: {cfg['model']['name']}")
print(f"  Batch size: {cfg['training']['batch_size']}")
print(f"  Effective batch: {cfg['training']['batch_size'] * cfg['training']['gradient_accumulation_steps']}")
print(f"  Epochs: {cfg['training']['epochs']}")
print(f"  LoRA rank: {cfg['training']['lora_r']}")
print(f"  Augmentation: {'✓' if cfg['augmentation']['enabled'] else '✗'}")

print("\n" + "=" * 70)
print("✓ Setup complete - ready to train")
print("=" * 70)

## 6. (Optional) Adjust Config for T4 GPU

If running on T4 (16GB VRAM) instead of A100, reduce batch size to avoid OOM.

In [ ]:
# Uncomment to auto-adjust for T4

# import torch
# import yaml

# gpu_name = torch.cuda.get_device_name(0)
# vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

# if vram_gb < 30:  # T4 or smaller
#     print(f"🔧 Detected {gpu_name} ({vram_gb:.0f}GB) - adjusting config for lower VRAM")
    
#     with open("config.yaml") as f:
#         cfg = yaml.safe_load(f)
    
#     cfg['training']['batch_size'] = 2
#     cfg['training']['gradient_accumulation_steps'] = 8  # maintain effective batch=16
#     cfg['training']['lora_r'] = 32  # reduce rank
    
#     with open("config.yaml", "w") as f:
#         yaml.dump(cfg, f)
    
#     print("✓ Config adjusted: batch_size=2, lora_r=32")
# else:
#     print(f"✓ {gpu_name} detected - using default config")

print("Using default config (A100 optimized)")

## 7. Train Model 🚀

**Expected time:**
- A100 40GB: 6-8 hours
- A100 80GB: 6-8 hours  
- T4 16GB: 12-15 hours (with adjusted batch size)

**Memory usage:**
- A100: ~45-55 GB
- T4: ~14-15 GB (with batch_size=2, lora_r=32)

The training will:
- Load Qwen2-VL-7B-Instruct (~15 GB)
- Apply LoRA fine-tuning (rank 64)
- Use image augmentation (blur, noise, contrast, rotation)
- Evaluate every 100 steps
- Save best model based on eval loss

**Note:** Colab may disconnect after 12 hours. For long training, consider:
1. Mounting Drive to save checkpoints
2. Using smaller LoRA rank or fewer epochs
3. Enabling `save_steps` to save intermediate checkpoints

In [ ]:
import time
start = time.time()

print("🚀 Starting training...\n")
!python train.py

elapsed = (time.time() - start) / 3600
print(f"\n✓ Training completed in {elapsed:.1f} hours")

## 8. Generate Submission

Run inference on test set using the best checkpoint.

In [ ]:
print("🔮 Running inference on test set...\n")
!python inference.py

print("\n✓ Submission generated: ../../submission.csv")

## 9. Verify Submission

In [ ]:
import pandas as pd

submission = pd.read_csv("../../submission.csv")

print("=" * 70)
print("📊 SUBMISSION PREVIEW")
print("=" * 70)
print(submission.head(10))

print("\n" + "=" * 70)
print("📈 SUBMISSION STATS")
print("=" * 70)
print(f"Total predictions: {len(submission)}")
print(f"Empty predictions: {(submission['Target'] == '').sum()}")
print(f"Avg text length: {submission['Target'].str.len().mean():.1f} chars")
print(f"Min text length: {submission['Target'].str.len().min():.0f} chars")
print(f"Max text length: {submission['Target'].str.len().max():.0f} chars")

print("\n✓ Submission ready: ../../submission.csv")
print("=" * 70)

## 10. Download Files

Download submission and checkpoints to your local machine.

In [ ]:
from google.colab import files

# Download submission
print("Downloading submission.csv...")
files.download("../../submission.csv")

# Optional: Download best checkpoint (large file)
# print("\nPacking best checkpoint...")
# !tar -czf checkpoint_best.tar.gz outputs/qwen2vl-7b-run1/best/
# files.download("checkpoint_best.tar.gz")

print("\n✓ Download complete")

## 11. (Optional) Sample Predictions Visualization

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random

# Sample 3 random predictions
samples = submission.sample(3)

fig, axes = plt.subplots(3, 1, figsize=(15, 12))

for ax, (_, row) in zip(axes, samples.iterrows()):
    img_path = f"../../dataset/images/{row['ID']}.jpg"
    img = Image.open(img_path)
    
    ax.imshow(img)
    ax.set_title(f"Prediction: {row['Target'][:100]}...", fontsize=9, pad=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

---

## 🎯 Next Steps for Improvement

### 1. Error Analysis
- Review predictions on validation set
- Identify common error patterns (missing words, wrong letters)
- Adjust augmentation strategy based on errors

### 2. Hyperparameter Tuning
- **Increase epochs:** Try 7-10 epochs
- **Learning rate:** Experiment with 1e-5 to 3e-5
- **LoRA rank:** Try 32, 64, 128 (higher = more capacity)
- **Augmentation:** Adjust probabilities based on error patterns

### 3. Ensemble Strategy
- Train multiple models with different seeds
- Train TrOCR-large for architectural diversity
- Combine predictions with weighted voting
- Test-time augmentation (3-5 passes per image)

### 4. Post-processing
- Historical spelling correction
- Language model filtering (GPT for common phrases)
- Common pattern detection (dates, names, etc.)

### 5. Advanced Techniques
- Curriculum learning (train on easy samples first)
- Focal loss for hard examples
- Self-training on test set pseudo-labels

---

## 📊 Expected Performance

| Configuration | WER | CER | Combined Score |
|--------------|-----|-----|----------------|
| Baseline (no aug) | 15-20% | 5-8% | 10-14% |
| + Augmentation | 12-15% | 4-6% | 8-10.5% |
| + Ensemble (2-3 models) | 10-12% | 3-5% | 6.5-8.5% |
| + Post-processing | 8-10% | 2-4% | 5-7% |

**Target for top 10%:** Combined score < 8%  
**Target for top 3%:** Combined score < 6%

---

Good luck! 🚀